# Longitudinal trajectories, matched to the cross-sectional method

Applies the cross-sectional paper's measures, ROIs, and tests to the
longitudinal data. The only change is the unit of analysis: instead of one
session per subject, each subject contributes the **change between consecutive
sessions**, and the group comparisons are run on those change scores.

**Provenance.** Helpers (`perm_independent`, `perm_paired`, `cohens_d_*`,
`boot_ci_*`, `fdr_bh_by`, `lmm_omnibus`) are copied verbatim from
`D_liu/verified/05_stats_harmony.py`. The |LI| and rank codes are copied from
`marlene_grid.py` (lines 138-153). No new statistical approach is introduced.

**Inputs are the UNHARMONIZED extraction CSVs.** `rsa_v1_harmonized.csv` and
`univariate_v1_harmonized_sqrt.csv` carry one session per subject, so they
cannot support a longitudinal analysis. `rsa_v1.csv` and `univariate_v1.csv`
carry every session.

**Decisions recorded in config, not buried:**

| decision | setting | why |
|---|---|---|
| sub-108 ses-02 | **included** | registration confirmed working (fslstats, May 2026); the cross-sectional script's `EXCLUDE_SES` entry predates that fix |
| ComBat | **off** (`USE_COMBAT = False`) | a within-subject difference cancels the additive scanner term; the variance term does not, which is a stated limitation rather than something ComBat as configured would fix |
| sub-025, sub-064 | **dropped** | the only subjects whose scanner changes between sessions, so their change score is confounded |
| session pairing | all consecutive pairs | sub-004 has 5 sessions, sub-021 and sub-025 have 3 |
| hemispheres | controls both, patients intact only | same as cross-sectional |

Every printed number is produced by the cell above it. Nothing is carried over
from previous decks or handoff documents.

In [ ]:
# ── Imports and paths ───────────────────────────────────────────────────────
import sys, itertools, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2 as chi2_dist, pearsonr
from pathlib import Path

warnings.filterwarnings('ignore')

GIT  = Path('/user_data/csimmon2/git_repos/sym_pt')
D_LIU = GIT / 'D_liu'

UNIVAR_CSV = D_LIU / 'univariate_v1.csv'      # unharmonized: all sessions
RSA_CSV    = D_LIU / 'rsa_v1.csv'             # unharmonized: all sessions
SUB_INFO   = GIT / 'sub_info.csv'
SCANNER    = GIT / 'F_harmonization' / 'sub_info_scanner.csv'

OUT_CSV    = D_LIU / 'stats_results_longitudinal.csv'
OUT_COHORT = D_LIU / 'longitudinal_cohort.csv'

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
# Subject exclusions: identical to 05_stats_harmony.py.
EXCLUDE = ['sub-017', 'sub-091', 'sub-095', 'sub-096', 'sub-027', 'sub-084']

# NOTE: 05_stats_harmony.py also carries EXCLUDE_SES = [('sub-108', 2)].
# That is deliberately NOT applied here. sub-108 has only ses-01 and ses-02,
# so applying it would remove sub-108 from the longitudinal analysis entirely.
EXCLUDE_SES = []

# Scanner changes between sessions in these two controls only; all six
# patients are single-scanner throughout.
DROP_SCANNER_SWITCHERS = ['sub-025', 'sub-064']

USE_COMBAT = False          # see header table
PAIR_AGG   = 'mean'         # 'mean' = one value per subject averaged over
                            # consecutive pairs; 'all' = every pair as a row

PRIMARY_ROIS = ['face_FFA', 'house_PPA_strict', 'object_LOC', 'word_VWFA']
CATEGORIES   = ['face', 'house', 'object', 'word']
PAIRS        = ['face-house', 'face-object', 'face-word',
                'house-object', 'house-word', 'object-word']

# ROI -> preferred category (marlene_grid.py PREF)
PREF = {'object_LOC': 'object', 'house_PPA': 'house',
        'house_PPA_strict': 'house', 'face_FFA': 'face', 'word_VWFA': 'word'}

# Control |LI| at z>2.33, copied from marlene_grid.py lines 138-142.
LI_ROI = {'word_VWFA': 0.389, 'face_FFA': 0.245,
          'object_LOC': 0.130, 'house_PPA': 0.124, 'house_PPA_strict': 0.124}
# A priori rank order: word > face > object = house. object and house tie
# because they swap rank at z>1.96 (marlene_grid.py lines 144-152).
RANK_ROI = {'word_VWFA': 4.0, 'face_FFA': 3.0,
            'object_LOC': 1.5, 'house_PPA': 1.5, 'house_PPA_strict': 1.5}
# binA: word+face vs house+object (marlene_grid.py SPLITS)
BINA_ROI = {'word_VWFA': 1, 'face_FFA': 1,
            'object_LOC': 0, 'house_PPA': 0, 'house_PPA_strict': 0}

N_PERM = 10000
N_BOOT = 5000
FDR_METHOD = 'bh'
RNG = np.random.RandomState(42)

print('ROIs      :', PRIMARY_ROIS)
print('exclusions:', EXCLUDE)
print('dropped   :', DROP_SCANNER_SWITCHERS)
print('ComBat    :', USE_COMBAT, '| pair aggregation:', PAIR_AGG)

## Helpers

Copied verbatim from `D_liu/verified/05_stats_harmony.py`. Do not edit here —
if these need changing, change them there and re-copy, so the two analyses
cannot silently diverge.

In [ ]:
def fdr_bh_by(pvals, method='bh', alpha=0.05):
    """Benjamini-Hochberg ('bh') or Benjamini-Yekutieli ('by') q-values."""
    p = np.asarray(pvals, float)
    valid = ~np.isnan(p)
    q = np.full_like(p, np.nan)
    pv = p[valid]; n = len(pv)
    if n == 0:
        return q, np.zeros_like(p, bool)
    order = np.argsort(pv); ranked = pv[order]
    c_n = np.sum(1.0 / np.arange(1, n + 1)) if method == 'by' else 1.0
    q_ranked = ranked * n * c_n / np.arange(1, n + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)
    q_valid = np.empty(n); q_valid[order] = q_ranked
    q[valid] = q_valid
    sig = np.zeros_like(p, bool); sig[valid] = q_valid <= alpha
    return q, sig

def cohens_d_independent(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def cohens_d_paired(diff):
    diff = np.asarray(diff, float); diff = diff[~np.isnan(diff)]
    if len(diff) < 2 or diff.std(ddof=1) == 0:
        return np.nan
    return diff.mean() / diff.std(ddof=1)

def perm_independent(a, b, n_perm=N_PERM, rng=RNG):
    """Two-tailed label-shuffle permutation on mean difference (a - b)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan
    obs = a.mean() - b.mean()
    pool = np.concatenate([a, b]); na = len(a)
    null = np.empty(n_perm)
    for i in range(n_perm):
        idx = rng.permutation(len(pool))
        null[i] = pool[idx[:na]].mean() - pool[idx[na:]].mean()
    p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p

def perm_paired(diff, n_perm=N_PERM, rng=RNG):
    """Two-tailed sign-flip permutation on the mean of paired differences."""
    diff = np.asarray(diff, float); diff = diff[~np.isnan(diff)]
    if len(diff) < 2:
        return np.nan, np.nan
    obs = diff.mean()
    null = np.empty(n_perm)
    for i in range(n_perm):
        signs = rng.choice([-1, 1], size=len(diff))
        null[i] = (diff * signs).mean()
    p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p

def boot_ci_independent(a, b, n_boot=N_BOOT, rng=RNG):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan
    ds = np.empty(n_boot)
    for i in range(n_boot):
        ds[i] = cohens_d_independent(rng.choice(a, len(a), replace=True),
                                     rng.choice(b, len(b), replace=True))
    ds = ds[~np.isnan(ds)]
    if len(ds) == 0:
        return np.nan, np.nan
    return float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))

def boot_ci_paired(diff, n_boot=N_BOOT, rng=RNG):
    diff = np.asarray(diff, float); diff = diff[~np.isnan(diff)]
    if len(diff) < 2:
        return np.nan, np.nan
    ds = np.empty(n_boot)
    for i in range(n_boot):
        ds[i] = cohens_d_paired(rng.choice(diff, len(diff), replace=True))
    ds = ds[~np.isnan(ds)]
    if len(ds) == 0:
        return np.nan, np.nan
    return float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))

def lmm_omnibus(df, value_col, factor_col, group_col, with_age=True):
    """value ~ factor*group + age + (1|subject); joint Wald chi2 on the
    interaction. Returns (chi2, df, p, mse)."""
    cols = ['subject_id', value_col, factor_col, group_col]
    use_age = with_age and 'age' in df.columns
    if use_age:
        cols = cols + ['age']
    d = df[cols].dropna().copy()
    d = d.rename(columns={value_col: 'y', factor_col: 'f', group_col: 'g'})
    if d['g'].nunique() < 2 or d['f'].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan
    if use_age and d['age'].nunique() < 2:
        use_age = False
    formula = 'y ~ C(f) * C(g) + age' if use_age else 'y ~ C(f) * C(g)'
    try:
        m = smf.mixedlm(formula, d, groups=d['subject_id']).fit(reml=True)
    except Exception:
        if not use_age:
            return np.nan, np.nan, np.nan, np.nan
        try:
            m = smf.mixedlm('y ~ C(f) * C(g)', d,
                            groups=d['subject_id']).fit(reml=True)
        except Exception:
            return np.nan, np.nan, np.nan, np.nan
    inter = [i for i, n in enumerate(m.params.index) if ':' in n]
    if not inter:
        return np.nan, np.nan, np.nan, float(m.scale)
    beta = m.params.values[inter]
    V = m.cov_params().values[np.ix_(inter, inter)]
    try:
        W = float(beta @ np.linalg.solve(V, beta))
    except np.linalg.LinAlgError:
        W = float(beta @ np.linalg.pinv(V) @ beta)
    dfree = len(inter)
    return W, dfree, float(chi2_dist.sf(W, dfree)), float(m.scale)

def row(**kw):
    base = dict(measure=None, model=None, comparison=None, hemi=None, level=None,
                omnibus_chi2=np.nan, omnibus_df=np.nan, omnibus_p=np.nan,
                mse=np.nan, gating=None, diff=np.nan, cohens_d=np.nan,
                p_perm=np.nan, q_fdr=np.nan, ci_lo=np.nan, ci_hi=np.nan,
                n_a=np.nan, n_b=np.nan, note='')
    base.update(kw)
    return base

print('helpers defined')

## Load, exclude, and build the longitudinal cohort

Age is merged per subject x session, so the LMM covariate is age at that
session rather than a single age per subject.

In [ ]:
def age_table():
    si = pd.read_csv(SUB_INFO)
    si['subject_id'] = si['sub'].astype(str)
    si['ses_num'] = pd.to_numeric(
        si['ses'].astype(str).str.extract(r'(\d+)', expand=False),
        errors='coerce').astype('Int64')
    return (si[['subject_id', 'ses_num', 'age']].dropna()
            .drop_duplicates(['subject_id', 'ses_num']))

def load(path):
    df = pd.read_csv(path)
    df = df[~df['subject_id'].isin(EXCLUDE)].copy()
    df['ses_num'] = pd.to_numeric(df['session'], errors='coerce').astype('Int64')
    for bad_sub, bad_ses in EXCLUDE_SES:
        df = df[~((df['subject_id'] == bad_sub) & (df['ses_num'] == bad_ses))]
    df = df[~df['subject_id'].isin(DROP_SCANNER_SWITCHERS)]
    df = df.drop(columns=[c for c in ('age',) if c in df.columns])
    df = df.merge(age_table(), on=['subject_id', 'ses_num'], how='left')
    # keep only subjects with >= 2 sessions
    n = df.groupby('subject_id')['ses_num'].transform('nunique')
    return df[n >= 2].copy()

uni = load(UNIVAR_CSV)
rsa = load(RSA_CSV)

# The univariate file carries 9 nonOTC patients with >=2 sessions that the RSA
# file does not (sub-007, 045, 047, 049, 070, 072, 073, 081, 086). They are
# never entered into a group test below, which selects group=='OTC' or
# status=='control'. They are reported here because a nonOTC group is the
# cleanest available control for acute surgical effects, and adding it is a
# live option rather than something to drop silently.
nonotc = (uni[uni.group == 'nonOTC'].groupby('subject_id')
            .agg(n_ses=('ses_num','nunique')).reset_index())
nonotc = nonotc[nonotc.n_ses >= 2]
print('nonOTC patients with >=2 sessions, univariate only:',
      nonotc.subject_id.tolist())
print()

cohort = (rsa.groupby('subject_id')
            .agg(group=('group','first'), status=('status','first'),
                 surgery_side=('surgery_side','first'),
                 intact_hemi=('intact_hemi','first'),
                 n_ses=('ses_num','nunique'),
                 sessions=('ses_num', lambda s: sorted(set(s.dropna()))),
                 ages=('age', lambda s: sorted(set(s.dropna()))))
            .reset_index().sort_values(['group','subject_id']))
cohort.to_csv(OUT_COHORT, index=False)
print(cohort.to_string(index=False))
print()
pt = cohort[cohort.group == 'OTC']
print('patients:', len(pt),
      '| LH-intact', (pt.intact_hemi == 'left').sum(),
      '| RH-intact', (pt.intact_hemi == 'right').sum())
print('controls:', (cohort.status == 'control').sum())

## Change scores

For every subject, hemisphere, and ROI, each consecutive session pair gives one
change score. Signed measures use `later - earlier`. Peak location uses
Euclidean distance in native space, which is unsigned by construction, so it is
drift in mm rather than a signed change.

With `PAIR_AGG = 'mean'` a subject with more than two sessions contributes the
average of its consecutive-pair values, so every subject carries equal weight
in the group tests regardless of how many sessions it has.

In [ ]:
def consecutive_pairs(sessions):
    s = sorted(set(int(x) for x in sessions))
    return list(zip(s[:-1], s[1:]))

def signed_change(df, value_col, roi_col='category', extra_key=None):
    """Long frame of change scores, one row per
    subject x hemi x roi [x extra_key] x session-pair."""
    keys = ['subject_id', 'hemi', roi_col] + ([extra_key] if extra_key else [])
    out = []
    meta = df.groupby('subject_id').agg(
        group=('group','first'), status=('status','first'),
        intact_hemi=('intact_hemi','first')).to_dict('index')
    for key, g in df.groupby(keys, dropna=True):
        sub = key[0]
        byses = g.set_index('ses_num')
        for s0, s1 in consecutive_pairs(byses.index):
            if s0 not in byses.index or s1 not in byses.index:
                continue
            v0, v1 = byses.loc[s0, value_col], byses.loc[s1, value_col]
            if np.isscalar(v0) is False or np.isscalar(v1) is False:
                continue
            if pd.isna(v0) or pd.isna(v1):
                continue
            rec = dict(zip(keys, key))
            rec.update(dict(ses_a=s0, ses_b=s1, delta=float(v1) - float(v0),
                            age=float(byses.loc[s1, 'age'])
                                if not pd.isna(byses.loc[s1, 'age']) else np.nan,
                            **meta[sub]))
            out.append(rec)
    return pd.DataFrame(out)

def peak_drift(df):
    """Euclidean distance (mm) between consecutive sessions' peak voxels."""
    cols = ['peak_x_native', 'peak_y_native', 'peak_z_native']
    out = []
    meta = df.groupby('subject_id').agg(
        group=('group','first'), status=('status','first'),
        intact_hemi=('intact_hemi','first')).to_dict('index')
    for (sub, hemi, roi), g in df.groupby(['subject_id','hemi','category']):
        byses = g.drop_duplicates('ses_num').set_index('ses_num')
        for s0, s1 in consecutive_pairs(byses.index):
            a = byses.loc[s0, cols].astype(float).values
            b = byses.loc[s1, cols].astype(float).values
            if np.isnan(a).any() or np.isnan(b).any():
                continue
            out.append(dict(subject_id=sub, hemi=hemi, category=roi,
                            ses_a=s0, ses_b=s1,
                            delta=float(np.linalg.norm(b - a)),
                            age=float(byses.loc[s1, 'age'])
                                if not pd.isna(byses.loc[s1, 'age']) else np.nan,
                            **meta[sub]))
    return pd.DataFrame(out)

def collapse(ch):
    """Average over consecutive pairs if PAIR_AGG == 'mean'."""
    if PAIR_AGG != 'mean':
        return ch
    keys = [c for c in ['subject_id','hemi','category','pair'] if c in ch.columns]
    agg = {'delta':'mean','age':'mean','group':'first','status':'first',
           'intact_hemi':'first'}
    return ch.groupby(keys, as_index=False).agg(agg)

# unique-row frames (one row per subject x ses x hemi x roi) for univariate
uni_u = uni.drop_duplicates(['subject_id','ses_num','hemi','category'])
rsa_u = rsa.drop_duplicates(['subject_id','ses_num','hemi','category'])

CH = {}
CH['peak_location']   = collapse(peak_drift(uni_u))
CH['peak_selectivity']= collapse(signed_change(uni_u, 'peak_z'))
CH['voxel_count']     = collapse(signed_change(uni_u, 'volume'))
CH['sum_selectivity'] = collapse(signed_change(uni_u, 'sum_selec_norm'))
CH['pref_similarity'] = collapse(signed_change(rsa_u, 'liu_distinctiveness'))
CH['btw_similarity']  = collapse(signed_change(
                            rsa.drop_duplicates(
                                ['subject_id','ses_num','hemi','category','pair']),
                            'fisher_r', extra_key='pair'))

for k, v in CH.items():
    print(f'{k:18s} rows {len(v):5d} | subjects {v.subject_id.nunique():3d}')

## Geometry preservation

The one measure with no cross-sectional counterpart, because it is defined over
time: the Pearson correlation between a subject's six between-category
similarities at session A and the same six at session B, within an ROI. Higher
means the internal representational structure is more preserved across sessions.
This is the `long_pt` measure, computed here on the cross-sectional ROI
definition instead of the Desikan-Killiany parcels.

In [ ]:
def geometry_preservation(df):
    out = []
    meta = df.groupby('subject_id').agg(
        group=('group','first'), status=('status','first'),
        intact_hemi=('intact_hemi','first')).to_dict('index')
    for (sub, hemi, roi), g in df.groupby(['subject_id','hemi','category']):
        piv = g.pivot_table(index='ses_num', columns='pair', values='fisher_r')
        piv = piv.reindex(columns=PAIRS)
        for s0, s1 in consecutive_pairs(piv.index):
            a, b = piv.loc[s0].values, piv.loc[s1].values
            if np.isnan(a).any() or np.isnan(b).any():
                continue
            r = pearsonr(a, b)[0]
            age = g[g.ses_num == s1]['age'].mean()
            out.append(dict(subject_id=sub, hemi=hemi, category=roi,
                            ses_a=s0, ses_b=s1, delta=float(r),
                            age=float(age) if not pd.isna(age) else np.nan,
                            **meta[sub]))
    return pd.DataFrame(out)

CH['geometry_preservation'] = collapse(geometry_preservation(rsa))
g = CH['geometry_preservation']
print('geometry rows', len(g), '| subjects', g.subject_id.nunique())
print()
print(g[g.category.isin(PRIMARY_ROIS)]
      .pivot_table(index=['status','subject_id'], columns='category',
                   values='delta').round(3).to_string())

## Group comparisons

The cross-sectional four-comparison sequence, run on change scores:

1. controls, left versus right hemisphere (paired)
2. each patient group versus matched-hemisphere controls (independent)
3. patients versus patients (independent)
4. lateralization — LMM omnibus across ROIs, plus the |LI| gradient, rank
   order, and the word+face versus house+object split

FDR families match the cross-sectional script: across ROIs within a comparison.

In [ ]:
def compare_measure(ch, measure, rois=PRIMARY_ROIS, results=None):
    res = [] if results is None else results

    def vals(mask, roi, hemi):
        d = ch[(mask) & (ch.category == roi) & (ch.hemi == hemi)]
        return d['delta'].dropna().values

    is_ctrl = ch.status == 'control'
    is_pt   = ch.group == 'OTC'

    # ---- 1. controls, left vs right (paired within subject) ----------------
    pr, pv = [], []
    for roi in rois:
        w = (ch[is_ctrl & (ch.category == roi)]
             .pivot_table(index='subject_id', columns='hemi', values='delta'))
        if not {'l','r'}.issubset(w.columns):
            continue
        diff = (w['l'] - w['r']).dropna().values
        if len(diff) < 2:
            continue
        obs, p = perm_paired(diff)
        lo, hi = boot_ci_paired(diff)
        pr.append(row(measure=measure, model='ctrl_L_vs_R',
                      comparison='control_lateralization', hemi='l-r', level=roi,
                      diff=obs, cohens_d=cohens_d_paired(diff), p_perm=p,
                      ci_lo=lo, ci_hi=hi, n_a=len(diff), n_b=len(diff)))
        pv.append(p)
    if pv:
        q, _ = fdr_bh_by(pv, FDR_METHOD)
        for r_, qq in zip(pr, q): r_['q_fdr'] = qq
    res += pr

    # ---- 2. each patient group vs matched-hemisphere controls --------------
    for hemi, label in [('l','LH-intact'), ('r','RH-intact')]:
        pr, pv = [], []
        for roi in rois:
            a = vals(is_pt & (ch.intact_hemi == {'l':'left','r':'right'}[hemi]),
                     roi, hemi)
            b = vals(is_ctrl, roi, hemi)
            if len(a) < 2 or len(b) < 2:
                continue
            obs, p = perm_independent(a, b)
            lo, hi = boot_ci_independent(a, b)
            pr.append(row(measure=measure, model=f'{label}_pt_vs_ctrl',
                          comparison='patient_vs_control', hemi=hemi, level=roi,
                          diff=obs, cohens_d=cohens_d_independent(a, b),
                          p_perm=p, ci_lo=lo, ci_hi=hi, n_a=len(a), n_b=len(b)))
            pv.append(p)
        if pv:
            q, _ = fdr_bh_by(pv, FDR_METHOD)
            for r_, qq in zip(pr, q): r_['q_fdr'] = qq
        res += pr

    # ---- 3. patients vs patients ------------------------------------------
    pr, pv = [], []
    for roi in rois:
        a = vals(is_pt & (ch.intact_hemi == 'left'),  roi, 'l')
        b = vals(is_pt & (ch.intact_hemi == 'right'), roi, 'r')
        if len(a) < 2 or len(b) < 2:
            continue
        obs, p = perm_independent(a, b)
        lo, hi = boot_ci_independent(a, b)
        pr.append(row(measure=measure, model='LH_pt_vs_RH_pt',
                      comparison='patient_vs_patient', hemi='intact', level=roi,
                      diff=obs, cohens_d=cohens_d_independent(a, b),
                      p_perm=p, ci_lo=lo, ci_hi=hi, n_a=len(a), n_b=len(b)))
        pv.append(p)
    if pv:
        q, _ = fdr_bh_by(pv, FDR_METHOD)
        for r_, qq in zip(pr, q): r_['q_fdr'] = qq
    res += pr
    return res

def lateralization_tests(ch, measure, rois=PRIMARY_ROIS, results=None):
    """LMM omnibus over ROIs, then the three a priori codings as OLS slopes
    on the ROI modifier. Same hypotheses as the cross-sectional grid."""
    res = [] if results is None else results
    sub = ch[ch.category.isin(rois)].copy()

    specs = [('gradient_absLI', LI_ROI), ('rank_order', RANK_ROI),
             ('sym_vs_asym', BINA_ROI)]

    comparisons = [
        ('ctrl_L_vs_R',        (sub.status=='control') & (sub.hemi=='l'),
                               (sub.status=='control') & (sub.hemi=='r')),
        ('LH-intact_pt_vs_ctrl',(sub.group=='OTC') & (sub.intact_hemi=='left'),
                               (sub.status=='control') & (sub.hemi=='l')),
        ('RH-intact_pt_vs_ctrl',(sub.group=='OTC') & (sub.intact_hemi=='right'),
                               (sub.status=='control') & (sub.hemi=='r')),
        ('LH_pt_vs_RH_pt',     (sub.group=='OTC') & (sub.intact_hemi=='left'),
                               (sub.group=='OTC') & (sub.intact_hemi=='right')),
    ]

    for name, mask_a, mask_b in comparisons:
        d = pd.concat([sub[mask_a].assign(_g='A'), sub[mask_b].assign(_g='B')])
        if d['_g'].nunique() < 2 or d.category.nunique() < 2:
            continue
        chi2, dfree, p, mse = lmm_omnibus(d, 'delta', 'category', '_g')
        res.append(row(measure=measure, model=name, comparison='lateralization',
                       level='omnibus_ROI', omnibus_chi2=chi2, omnibus_df=dfree,
                       omnibus_p=p, mse=mse,
                       n_a=int(mask_a.sum()), n_b=int(mask_b.sum())))
        for spec_name, codes in specs:
            dd = d.copy()
            dd['mod'] = dd.category.map(codes)
            dd['gb'] = (dd._g == 'B').astype(int)
            dd = dd.dropna(subset=['delta','mod'])
            if dd.empty or dd.gb.nunique() < 2:
                continue
            try:
                m = smf.mixedlm('delta ~ mod * gb', dd,
                                groups=dd['subject_id']).fit(reml=True)
                key = [i for i in m.params.index if i == 'mod:gb']
                if not key:
                    continue
                beta = float(m.params['mod:gb'])
                pval = float(m.pvalues['mod:gb'])
            except Exception:
                continue
            res.append(row(measure=measure, model=name,
                           comparison='lateralization', level=spec_name,
                           diff=beta, p_perm=pval,
                           n_a=int(mask_a.sum()), n_b=int(mask_b.sum()),
                           note='mixedlm interaction slope'))
    return res

print('comparison functions defined')

In [ ]:
# ── Run every measure ───────────────────────────────────────────────────────
results = []
for measure, ch in CH.items():
    if measure == 'btw_similarity':
        continue                      # handled separately: per-pair, below
    compare_measure(ch, measure, results=results)
    lateralization_tests(ch, measure, results=results)

print(len(results), 'rows from scalar measures')

In [ ]:
# ── Between-category similarity: per ROI over its six pairs ────────────────
# Mirrors the cross-sectional treatment, where the six pairs are kept separate
# and the omnibus is over pairs within an ROI rather than over ROIs.
ch = CH['btw_similarity']
is_ctrl = ch.status == 'control'; is_pt = ch.group == 'OTC'

comparisons = [
    ('ctrl_L_vs_R',          is_ctrl & (ch.hemi=='l'), is_ctrl & (ch.hemi=='r')),
    ('LH-intact_pt_vs_ctrl', is_pt & (ch.intact_hemi=='left'),  is_ctrl & (ch.hemi=='l')),
    ('RH-intact_pt_vs_ctrl', is_pt & (ch.intact_hemi=='right'), is_ctrl & (ch.hemi=='r')),
    ('LH_pt_vs_RH_pt',       is_pt & (ch.intact_hemi=='left'),  is_pt & (ch.intact_hemi=='right')),
]

for roi in PRIMARY_ROIS:
    for name, ma, mb in comparisons:
        d = pd.concat([ch[ma & (ch.category==roi)].assign(_g='A'),
                       ch[mb & (ch.category==roi)].assign(_g='B')])
        if d.empty or d['_g'].nunique() < 2 or d['pair'].nunique() < 2:
            continue
        chi2, dfree, p, mse = lmm_omnibus(d, 'delta', 'pair', '_g')
        results.append(row(measure='btw_similarity', model=name,
                           comparison='pair_omnibus', level=roi,
                           omnibus_chi2=chi2, omnibus_df=dfree, omnibus_p=p,
                           mse=mse, n_a=int((d._g=='A').sum()),
                           n_b=int((d._g=='B').sum())))
        # per-pair contrasts, FDR across the six
        pr, pv = [], []
        for pair in PAIRS:
            a = d[(d._g=='A') & (d.pair==pair)]['delta'].dropna().values
            b = d[(d._g=='B') & (d.pair==pair)]['delta'].dropna().values
            if len(a) < 2 or len(b) < 2:
                continue
            obs, pp = perm_independent(a, b)
            lo, hi = boot_ci_independent(a, b)
            pr.append(row(measure='btw_similarity', model=name,
                          comparison='per_pair', level=f'{roi}|{pair}',
                          diff=obs, cohens_d=cohens_d_independent(a, b),
                          p_perm=pp, ci_lo=lo, ci_hi=hi,
                          n_a=len(a), n_b=len(b)))
            pv.append(pp)
        if pv:
            q, _ = fdr_bh_by(pv, FDR_METHOD)
            for r_, qq in zip(pr, q): r_['q_fdr'] = qq
        results.extend(pr)

print(len(results), 'rows total')

In [ ]:
# ── Write and summarize ────────────────────────────────────────────────────
out = pd.DataFrame(results)
out.to_csv(OUT_CSV, index=False)
print('wrote', OUT_CSV, '|', len(out), 'rows')
print()
sig = out[(out.q_fdr <= 0.05) | (out.omnibus_p <= 0.05) |
          ((out.comparison=='lateralization') & (out.p_perm <= 0.05))]
cols = ['measure','model','comparison','level','diff','cohens_d',
        'omnibus_chi2','omnibus_df','omnibus_p','p_perm','q_fdr','n_a','n_b']
print('--- rows reaching p/q <= .05 ---')
print(sig[cols].round(4).to_string(index=False) if len(sig) else 'none')

## Still to decide

1. **ComBat.** `USE_COMBAT` is a flag but no harmonized longitudinal input
   exists yet. To turn it on, `combat_05b` and `combat_06` have to be rerun
   without session selection so they emit every session.
2. **`PAIR_AGG`.** Set to `'mean'`. Switching to `'all'` treats each
   consecutive pair as an observation, which gives sub-004 five times the
   weight of a two-session subject and violates independence unless the
   subject random intercept absorbs it.
3. **sub-008.** The only hemispherectomy patient, and the leave-one-out
   outlier in the previous RDM-distance analysis. Included here; whether it
   stays is a decision for the paper, not for this notebook.
4. **First-versus-last** as an alternative to consecutive pairs, if the
   consecutive-pair version is noisy.